# Notebook — Statistical shape analysis: Procrustes, GPA, and shape PCA

The whole shape pipeline in one notebook:
1. Represent shapes as **corresponding pointsets** (we synthesise a dataset with
   known modes of variation, buried under random position / size / rotation).
2. **Procrustes-align** two shapes (remove translation, scale, rotation via SVD).
3. **Generalised Procrustes Analysis (GPA)** to recover the **mean shape**.
4. **PCA** on the aligned shapes to recover the **modes of variation**
   (mean ± 2σ) — i.e. a Point Distribution Model.

NumPy + Matplotlib only. A note at the end shows how to swap in the real
`hands2D.mat` landmarks from `Assignments/a4/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (10, 4)

# --- a base outline (closed curve of N corresponding landmarks) ---
N = 40
t = np.linspace(0, 2*np.pi, N, endpoint=False)
base  = np.stack([np.cos(t), 0.55*np.sin(t)], axis=1)            # (N,2) ellipse-ish

# --- two TRUE modes of variation (genuine shape changes) ---
mode1 = np.stack([np.zeros(N), 0.35*np.sin(2*t)], axis=1)        # up/down lobes
mode2 = np.stack([0.30*np.cos(t)*np.sin(t), np.zeros(N)], axis=1)  # left/right skew

def random_similarity(pts):
    """Apply a random translation + scale + rotation (the nuisance we must remove)."""
    ang = rng.uniform(0, 2*np.pi)
    s = rng.uniform(0.7, 1.4)
    R = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
    T = rng.uniform(-1.5, 1.5, size=2)
    return s * (pts @ R.T) + T

# --- generate M shapes = base + mode coefficients + noise, then a random pose ---
M = 24
c1 = rng.normal(0, 1.0, M)      # true coefficient for mode 1
c2 = rng.normal(0, 0.6, M)      # true coefficient for mode 2
shapes = np.array([
    random_similarity(base + c1[m]*mode1 + c2[m]*mode2 + rng.normal(0, 0.012, base.shape))
    for m in range(M)
])                              # (M, N, 2)

def plot_shape(ax, pts, **kw):
    p = np.vstack([pts, pts[:1]])      # close the curve
    ax.plot(p[:, 0], p[:, 1], **kw)

fig, ax = plt.subplots(figsize=(6, 6))
for m in range(M):
    plot_shape(ax, shapes[m], color=plt.cm.tab20(m % 20), lw=1, alpha=0.7)
ax.set_aspect('equal'); ax.set_title(f'{M} raw shapes — different position, size, rotation')
plt.show()

## 1. Procrustes alignment of two shapes (Umeyama / Kabsch)

To compare two shapes we first remove translation, scale, and rotation. The
optimal rotation comes from the **SVD of the cross-covariance** (Kabsch /
Umeyama), with a sign fix so we get a true rotation (`det = +1`) and not a
reflection. The optimal scale follows from the singular values.

In [ ]:
def procrustes_align(src, dst):
    """Align src onto dst (translation + scale + rotation). Returns transformed src.
    Standard Umeyama solution: rotation from SVD of the cross-covariance,
    with a determinant sign-fix to forbid reflections, then optimal scale."""
    mu_a, mu_b = src.mean(0), dst.mean(0)
    A, B = src - mu_a, dst - mu_b
    n, D = src.shape
    var_a = (A**2).sum() / n
    Sigma = (B.T @ A) / n                       # D x D cross-covariance
    U, s_vals, Vt = np.linalg.svd(Sigma)
    S = np.eye(D)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[-1, -1] = -1.0                        # forbid reflection
    R = U @ S @ Vt                              # rotation (det +1)
    scale = (s_vals * np.diag(S)).sum() / var_a
    aligned = scale * (A @ R.T) + mu_b
    return aligned

# Demo: align shape 1 onto shape 0
aligned1 = procrustes_align(shapes[1], shapes[0])

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
plot_shape(ax[0], shapes[0], color='k', lw=2, label='target (shape 0)')
plot_shape(ax[0], shapes[1], color='tab:red', lw=1, label='source (shape 1)')
ax[0].set_title('before Procrustes'); ax[0].legend(); ax[0].set_aspect('equal')
plot_shape(ax[1], shapes[0], color='k', lw=2, label='target (shape 0)')
plot_shape(ax[1], aligned1, color='tab:green', lw=1, label='shape 1 aligned')
ax[1].set_title('after Procrustes'); ax[1].legend(); ax[1].set_aspect('equal')
plt.tight_layout(); plt.show()

## 2. The mean shape via Generalised Procrustes Analysis (GPA)

GPA finds the mean shape by **alternating minimisation**: put every shape in
preshape space (centre + unit norm), then repeat — (a) align all shapes to the
current mean, (b) re-average and re-normalise. Watch the cloud of outlines
collapse onto a clean mean.

In [ ]:
def to_preshape(z):
    z = z - z.mean(0)                 # centroid -> origin
    return z / np.linalg.norm(z)      # unit norm (size)

def gpa(shapes, n_iter=10):
    pre = np.array([to_preshape(s) for s in shapes])
    mean = pre[0].copy()
    for _ in range(n_iter):
        aligned = np.array([procrustes_align(p, mean) for p in pre])
        mean = to_preshape(aligned.mean(0))     # average, then project back to unit norm
    aligned = np.array([procrustes_align(p, mean) for p in pre])
    return mean, aligned

mean_shape, aligned = gpa(shapes)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
for m in range(M):
    plot_shape(ax[0], to_preshape(shapes[m]), color=plt.cm.tab20(m % 20), lw=0.8, alpha=0.6)
ax[0].set_title('preshape (centred + scaled) — still rotated'); ax[0].set_aspect('equal')
for m in range(M):
    plot_shape(ax[1], aligned[m], color='0.7', lw=0.6, alpha=0.8)
plot_shape(ax[1], mean_shape, color='crimson', lw=2.5)
ax[1].set_title('after GPA: aligned shapes + mean (red)'); ax[1].set_aspect('equal')
plt.tight_layout(); plt.show()

## 3. Modes of variation via PCA (the Point Distribution Model)

With the shapes aligned, the leftover scatter **is** the shape variability.
Flatten each aligned shape to a vector, take the sample covariance, and its
eigenvectors are the **modes of variation**. We expect ~2 dominant modes (we
built the data with two), and walking the mean along each mode (mean ± 2σ)
should reveal the up/down-lobe and left/right-skew deformations.

In [ ]:
X = aligned.reshape(M, -1)              # (M, 2N) shape vectors
mu = X.mean(0)
Xc = X - mu
C = (Xc.T @ Xc) / (M - 1)              # shape covariance (2N x 2N)
w, V = np.linalg.eigh(C)               # ascending
order = np.argsort(w)[::-1]
w, V = w[order], V[:, order]           # descending: w[k] = variance of mode k

# spectrum
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(np.arange(1, 11), w[:10] / w.sum())
ax[0].set_xlabel('mode'); ax[0].set_ylabel('fraction of variance')
ax[0].set_title('shape PCA spectrum (first few dominate)')
ax[1].plot(np.arange(1, 11), np.cumsum(w[:10]) / w.sum(), 'o-')
ax[1].set_ylim(0, 1.02); ax[1].set_xlabel('# modes'); ax[1].set_ylabel('cumulative variance')
ax[1].set_title('cumulative variance')
plt.tight_layout(); plt.show()

# mean +/- 2 sigma along the top 2 modes
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
for k in range(2):
    sd = np.sqrt(w[k])
    for b, ls, col in [(-2, '--', 'tab:blue'), (0, '-', 'k'), (2, '--', 'tab:red')]:
        shp = (mu + b * sd * V[:, k]).reshape(-1, 2)
        plot_shape(ax[k], shp, color=col, ls=ls, lw=(2.4 if b == 0 else 1.3),
                   label=('mean' if b == 0 else f'{b:+d}\u03c3'))
    ax[k].set_aspect('equal'); ax[k].legend()
    ax[k].set_title(f'Mode {k+1}: {100*w[k]/w.sum():.0f}% of variance')
plt.tight_layout(); plt.show()

## Exercises

1. **Generate new shapes:** sample shape parameters `b ~ N(0, w[:2])` and render
   `mean + Σ b_k √w_k V[:,k]`. Do they look like plausible members of the family?
2. **Compactness:** how many modes capture 95% of the variance? Relate it to the
   two modes you built in.
3. **Reflections:** remove the determinant sign-fix in `procrustes_align` and
   show a case where a shape gets flipped (wrongly aligned).
4. **Procrustes distance matrix:** compute pairwise Procrustes distances between
   all shapes and visualise the M×M matrix.
5. **Outliers / robust mean:** add one corrupted shape and watch the GPA mean
   drift; then replace the squared-distance average with a median-like update.

## Real data (optional — needs SciPy just to load the `.mat`)

```python
import scipy.io
d = scipy.io.loadmat('../../Assignments/a4/data/hands2D.mat')
print(d.keys())          # find the array of hand landmarks
# reshape to (M, N, 2), then run gpa(...) and the PCA above unchanged.
```

The hand dataset's first modes are typically **finger spread** and **thumb
angle** — exactly the Point Distribution Model used in Active Shape Models.